# 07 — Cascada de casos y targets (Y1 intensificación, Y2 persistencia)

Punto de entrada único de esta ronda (D24, `docs/decisiones_metodologicas.md`):
antes de ajustar ningún modelo, cuenta **cuántos casos reales** quedan para
cada target en cada paso del pipeline, y en cada fold temporal. Puramente
descriptivo/de construcción de etiquetas -- no entrena nada.

**Lee** `data/02_nucleo/{hogar,individual}`, `data/01_armonizado/{hogar,individual}`
(para `CAT_INAC`/`CH10`/`NIVEL_ED`, fuera de `ESQUEMA_NUCLEO`, mismo patrón ad
hoc que usa `06_poblacion_riesgo.ipynb`), `data/03_panel/pares_h1`/`pares_h4`
(capa ya construida, no se reconstruye) y `data/meta/trayectorias_completitud_cohorte.csv`
(para el flag D13). **No re-corre** ninguna notebook 01-06 ni toca ningún
módulo de `src/` -- todo lo nuevo vive en `src/cascada.py` (importado, no
editado acá) y en esta notebook.

**Escribe** `data/meta/cascada_trimestral.csv`, `vivienda_frecuencias.csv`
(+ `_aglomerado`) y `cascada_folds.csv`, más el Parquet intermedio
`data/07_targets/pares_targets/` (gitignored, igual que `0[1-9]_*`).

**Bloqueado esta ronda**: `grilla_3x3_trimestral.csv` (eje ingreso × trabajo)
queda pendiente de que Iván valide el deflactor FACPCE -- ver la entrada D24
correspondiente. El eje "trabajo" de 3 niveles ya queda calculado y guardado
en el Parquet intermedio para cuando se destrabe.

**Y1/Y2, decisión de Iván 2026-10-05**: **no se excluye** el ingreso no
declarado por default -- Y1/Y2 se calculan sobre todo par con
`identidad_confirmada==True` (más el filtro propio de cada target: ≥1 activo
estable en ambas puntas para Y1, `activos_sin_empleo_formal` en `t` para
Y2). La versión que SÍ excluye ingreso no declarado se reporta en paralelo
(columnas `..._excl_ingreso`), reservada para cuando se arme la grilla 3×3
(eje ingreso) -- no para dimensionar Y1/Y2 en general. Los pares con
ingreso no declarado en `t` o `t+h` (detectable solo en la era regular, D3)
quedan con `grupo_ingreso_t="no_declarado"` en el Parquet intermedio, en vez
de perderse o mezclarse con un quintil real.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd

import panel
import pipeline_meta as pm
from armonizacion import trimestres_publicados, es_historico, flags_trimestre
from intensificacion import (
    TRATAMIENTOS_INDEPENDIENTE, TRATAMIENTO_INDEPENDIENTE_DEFAULT,
    emparejar_activos, calcular_share_precario,
)
from eventos_hogar import calcular_alta_activo_menor, calcular_se_jubila
from perfil_hogar import calcular_componentes_ancla, calcular_grupo_ingreso
from tipado_nucleo import ESQUEMA_NUCLEO
from cascada import (
    tiene_destino_valido, clasificar_vinculacion_trimestre, clasificar_trabajo_lexicografico,
    calcular_persistencia, construir_fila_cascada, aglomerados_excluidos_d1, clasificar_fold,
    frecuencias_vivienda,
)

TRIMESTRES: list[tuple[int, int]] | None = None
REHACER: bool = False

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
PANEL_DIR = REPO / "data" / "03_panel"
META = REPO / "data" / "meta"
TARGETS = REPO / "data" / "07_targets" / "pares_targets"

TODOS_LOS_TRIMESTRES = trimestres_publicados()
trimestres_a_procesar = TRIMESTRES if TRIMESTRES is not None else TODOS_LOS_TRIMESTRES
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
SUFIJO = "_parcial" if ES_CORRIDA_PARCIAL else ""
COMMIT_GIT = pm.obtener_commit_git(REPO)
t0_proceso = time.time()
print(f"{len(trimestres_a_procesar)} trimestres de origen a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else ""))


`COLUMNAS_HOGAR` sigue el mismo set que `06_poblacion_riesgo.ipynb` (incluye
los insumos del ancla habitacional, D15). `COLUMNAS_VIVIENDA` se toma de
`tipado_nucleo.ESQUEMA_NUCLEO` (las variables `base="hogar"` con prefijo
`II`/`IV`) en vez de volver a tipear la lista a mano -- es la misma fuente de
verdad que usa la capa tipada.

`CAT_INAC`/`CH10` no están en `ESQUEMA_NUCLEO` (D19) -- `pipeline_meta.cargar_individual`
ya trae `NIVEL_ED` ad hoc desde `01_armonizado`; acá se agregan además esas
dos, mismo patrón, sin tocar `pipeline_meta.py`.

In [ ]:
COLUMNAS_HOGAR = ["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", "DECCFR", "PONDIH",
                  "II1", "II2", "II5", "II5_1", "IV1", "IV3", "IV8", "IV10", "IV12_1", "IV12_2", "IV12_3"]
COLUMNAS_INDIVIDUAL = ["CODUSU", "NRO_HOGAR", "COMPONENTE", "ANO4", "TRIMESTRE",
                        "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H"]
COLUMNAS_VIVIENDA = sorted(v for v, meta in ESQUEMA_NUCLEO.items()
                           if meta["base"] == "hogar" and (v.startswith("II") or v.startswith("IV")))

COHORTE_D13 = pd.read_csv(META / "trayectorias_completitud_cohorte.csv")
ORIGENES_D13 = set(
    COHORTE_D13.loc[COHORTE_D13["corte_estructural_d13"] == True, ["ANO4_t0", "TRIMESTRE_t0"]]
    .itertuples(index=False, name=None))
print(f"orígenes excluidos por D13 (h=4): {sorted(ORIGENES_D13)}")


def _agregar_cat_inac_ch10(individual: pd.DataFrame, anio: int, trimestre: int) -> pd.DataFrame:
    p = ARMONIZADO / "individual" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    extra = pd.read_parquet(p, columns=["CODUSU", "NRO_HOGAR", "COMPONENTE", "CAT_INAC", "CH10"])
    extra["CAT_INAC"] = pd.to_numeric(extra["CAT_INAC"], errors="coerce").astype("Int8")
    extra["CH10"] = pd.to_numeric(extra["CH10"], errors="coerce").astype("Int8")
    return individual.merge(extra, on=["CODUSU", "NRO_HOGAR", "COMPONENTE"], how="left")


def cargar_origen(anio: int, trimestre: int):
    hogar = pm.cargar_hogar(anio, trimestre, COLUMNAS_HOGAR, NUCLEO)
    individual = pm.cargar_individual(anio, trimestre, COLUMNAS_INDIVIDUAL, NUCLEO, ruta_armonizado=ARMONIZADO)
    if hogar is None or individual is None:
        return None, None
    individual = _agregar_cat_inac_ch10(individual, anio, trimestre)
    return hogar, individual


def calcular_ancla_y_grupo(hogar: pd.DataFrame, individual: pd.DataFrame) -> pd.DataFrame:
    idx = pd.MultiIndex.from_frame(hogar[["CODUSU", "NRO_HOGAR"]])
    h_idx = hogar.set_index(["CODUSU", "NRO_HOGAR"])
    n_miembros = individual.groupby(["CODUSU", "NRO_HOGAR"]).size().reindex(idx)
    ancla = calcular_componentes_ancla(h_idx, n_miembros)["ancla_con_deficit"]
    grupo = calcular_grupo_ingreso(hogar)
    grupo.index = idx
    return pd.DataFrame({"ancla_con_deficit": ancla.to_numpy(), "grupo_ingreso": grupo.to_numpy()}, index=idx)


## Sección 1 — destino válido por origen

`cascada.tiene_destino_valido` (D24): un origen `(anio_t, trimestre_t, h)`
queda `sin_destino_publicado` si `t+h` cae en un hueco de publicación,
después del último trimestre publicado, o cruza el corte de formato de
`CODUSU` entre la era histórica y la regular (D7) -- en los tres casos la
vinculación de ese origen no mide atrición real. Se clasifican los 86×2
orígenes posibles antes de calcular nada más.

In [ ]:
destino_valido = {}
for anio_t, trimestre_t in trimestres_a_procesar:
    for h in (1, 4):
        destino_valido[(anio_t, trimestre_t, h)] = tiene_destino_valido(anio_t, trimestre_t, h)

n_validos = sum(destino_valido.values())
n_invalidos = len(destino_valido) - n_validos
print(f"orígenes x h: {n_validos} con destino válido, {n_invalidos} sin destino publicado "
      f"(huecos de publicación, borde final de la serie, o cruce de era D7)")


## Secciones 2-4 — vinculación, Y1/Y2 (4 tratamientos de independientes) y eventos D19

Un solo loop por origen (evita releer `02_nucleo`/`01_armonizado` tres veces):
por cada `(anio_t, trimestre_t)` con destino válido en algún `h`, carga
`hogar_t`/`individual_t` una vez, y por cada `h` calcula, reusando sin editar
el código existente:

- **Vinculación** (`cascada.clasificar_vinculacion_trimestre`, D9/D24):
  vivienda y hogar, con los aglomerados de D1 excluidos del `%` de atrición
  cuando el destino es 2019T3 o 2020T3.
- **Y2** (`cascada.calcular_persistencia`, D14-b) y los eventos D19
  (`eventos_hogar.calcular_alta_activo_menor`/`calcular_se_jubila`) --
  no dependen del tratamiento de independientes, se calculan una sola vez.
- **Y1** (D16/D20 + la lectura lexicográfica de D24), una vez por cada uno
  de los 4 `TRATAMIENTOS_INDEPENDIENTE` (`emparejar_activos` +
  `calcular_share_precario` + `cascada.clasificar_trabajo_lexicografico`).
- `grupo_ingreso_t`: `perfil_hogar.calcular_grupo_ingreso` tal cual, salvo
  que se sobreescribe a `"no_declarado"` donde `ingreso_no_declarado_t` es
  `True` -- mismo criterio exacto que usa `cascada.construir_fila_cascada`
  para excluir en la versión `..._excl_ingreso`, así la categoría del
  Parquet intermedio (insumo de los folds) queda alineada 1 a 1 con esa
  exclusión.

El Parquet intermedio (`data/07_targets/pares_targets/`) solo guarda el
tratamiento default (`categoria_propia`) -- es el insumo de la Sección 6
(folds), que no necesita las 4 variantes para dimensionar los casos.

`REHACER=False` por default: reusa, por `(anio_t, trimestre_t, h)`, las
filas ya presentes en `cascada_trimestral.csv` si las 4 filas (una por
tratamiento) y la partición de `pares_targets` ya existen -- pura
existencia, igual criterio que el notebook 01. `REHACER=True` fuerza el
recálculo completo ignorando cualquier `cascada_trimestral.csv`/Parquet de
una corrida anterior (se usó así para la verificación posterior al cambio
de diseño de Y1/Y2 -- D24).

In [ ]:
CASCADA_CSV = META / f"cascada_trimestral{SUFIJO}.csv"
cascada_previa = pd.read_csv(CASCADA_CSV) if (not REHACER and CASCADA_CSV.exists()) else None


def filas_previas(anio_t: int, trimestre_t: int, h: int) -> list[dict] | None:
    if cascada_previa is None:
        return None
    filtro = ((cascada_previa["anio_t"] == anio_t) & (cascada_previa["trimestre_t"] == trimestre_t)
              & (cascada_previa["h"] == h))
    filas = cascada_previa[filtro]
    if len(filas) != len(TRATAMIENTOS_INDEPENDIENTE):
        return None
    return filas.to_dict("records")


TARGETS.mkdir(parents=True, exist_ok=True)
filas = []
n_reusados = 0
t0 = time.time()

for anio_t, trimestre_t in trimestres_a_procesar:
    hogar_t, individual_t = cargar_origen(anio_t, trimestre_t)
    if hogar_t is None:
        continue
    hogares_en_t = len(hogar_t[["CODUSU", "NRO_HOGAR"]].drop_duplicates())
    ancla_grupo_t = None

    for h in (1, 4):
        if not destino_valido[(anio_t, trimestre_t, h)]:
            for tratamiento in TRATAMIENTOS_INDEPENDIENTE:
                fila = construir_fila_cascada(hogares_en_t, "sin_destino_publicado", None, None)
                fila.update({"anio_t": anio_t, "trimestre_t": trimestre_t, "h": h,
                             "tratamiento_independiente": tratamiento})
                filas.append(fila)
            continue

        partition_path = TARGETS / f"ANO4_t={anio_t}" / f"TRIMESTRE_t={trimestre_t}" / f"h={h}" / "data.parquet"
        reusables = filas_previas(anio_t, trimestre_t, h)
        if not REHACER and reusables is not None and partition_path.exists():
            filas.extend(reusables)
            n_reusados += 1
            continue

        anio_th, trimestre_th = panel.sumar_trimestres(anio_t, trimestre_t, h)
        hogar_th, individual_th = cargar_origen(anio_th, trimestre_th)
        pares_path = PANEL_DIR / f"pares_h{h}" / f"ANO4={anio_t}" / f"TRIMESTRE={trimestre_t}" / "data.parquet"
        pares_hogar = pd.read_parquet(pares_path)

        if ancla_grupo_t is None:
            ancla_grupo_t = calcular_ancla_y_grupo(hogar_t, individual_t)

        vinculacion = clasificar_vinculacion_trimestre(
            hogar_t, hogar_th, pares_hogar,
            aglomerados_excluidos_atricion=aglomerados_excluidos_d1(anio_th, trimestre_th))

        base = pares_hogar[["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "identidad_confirmada",
                             "ingreso_declarado_t", "ingreso_declarado_th"]].copy()
        base["ingreso_no_declarado_t"] = ~base["ingreso_declarado_t"]
        base["ingreso_no_declarado_th"] = ~base["ingreso_declarado_th"]
        base = base.merge(ancla_grupo_t, left_on=["CODUSU", "NRO_HOGAR_t"], right_index=True, how="left")
        base = base.rename(columns={"ancla_con_deficit": "ancla_con_deficit_t", "grupo_ingreso": "grupo_ingreso_t"})
        base.loc[base["ingreso_no_declarado_t"] == True, "grupo_ingreso_t"] = "no_declarado"

        persistencia = calcular_persistencia(individual_t, individual_th, pares_hogar, h)
        base = base.merge(
            persistencia[["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "persiste_principal", "persiste_robustez"]],
            on=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"], how="left")

        n_miembros_t = individual_t.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros_t")
        n_miembros_th = individual_th.groupby(["CODUSU", "NRO_HOGAR"]).size().rename("n_miembros_th")
        base = base.merge(n_miembros_t, left_on=["CODUSU", "NRO_HOGAR_t"], right_index=True, how="left")
        base = base.merge(n_miembros_th, left_on=["CODUSU", "NRO_HOGAR_th"], right_index=True, how="left")
        base["cambio_tamanio_hogar"] = base["n_miembros_t"] != base["n_miembros_th"]

        eventos_menor = calcular_alta_activo_menor(individual_t, individual_th, pares_hogar, h)
        base = base.merge(eventos_menor[["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "n_alta_activo_menor"]],
                           on=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"], how="left")
        eventos_jub = calcular_se_jubila(individual_t, individual_th, pares_hogar, h)
        base = base.merge(eventos_jub, on=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"], how="left")

        corte_d13 = (anio_t, trimestre_t) in ORIGENES_D13 if h == 4 else False
        periodo_intervenido = flags_trimestre(anio_t, trimestre_t, [])["periodo_intervenido"]
        era = "historica" if es_historico(anio_t, trimestre_t) else "regular"
        en_holdout = (anio_t, trimestre_t) >= (2023, 1)
        destino_agl_d1 = bool(aglomerados_excluidos_d1(anio_th, trimestre_th))

        for tratamiento in TRATAMIENTOS_INDEPENDIENTE:
            emparejados = emparejar_activos(individual_t, individual_th, pares_hogar, h)
            share = calcular_share_precario(emparejados, tratamiento)
            share["trabajo_d16"] = np.where(
                share["share_precario_th"] > share["share_precario_t"], "empeora",
                np.where(share["share_precario_th"] < share["share_precario_t"], "mejora", "igual"))
            share["trabajo_lexicografico"] = clasificar_trabajo_lexicografico(emparejados, share)

            tabla_pares = base.merge(
                share[["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "n_activos_estables_t", "n_activos_estables_th",
                       "share_precario_t", "trabajo_d16", "trabajo_lexicografico"]],
                on=["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th"], how="left")

            fila = construir_fila_cascada(hogares_en_t, "valido", vinculacion, tabla_pares)
            fila.update({
                "anio_t": anio_t, "trimestre_t": trimestre_t, "h": h, "tratamiento_independiente": tratamiento,
                "corte_estructural_d13": corte_d13, "destino_aglomerado_faltante_d1": destino_agl_d1,
                "periodo_intervenido": periodo_intervenido, "era": era, "en_holdout": en_holdout,
                "n_alta_activo_menor": int(tabla_pares["n_alta_activo_menor"].fillna(0).sum()),
                "n_se_jubila": int(tabla_pares["n_se_jubila"].fillna(0).sum()),
                "n_cambio_tamanio_hogar": int(tabla_pares["cambio_tamanio_hogar"].fillna(False).sum()),
            })
            filas.append(fila)

            if tratamiento == TRATAMIENTO_INDEPENDIENTE_DEFAULT:
                destino = partition_path.parent
                destino.mkdir(parents=True, exist_ok=True)
                columnas_targets = ["CODUSU", "NRO_HOGAR_t", "NRO_HOGAR_th", "identidad_confirmada",
                                     "ingreso_no_declarado_t", "ingreso_no_declarado_th",
                                     "ancla_con_deficit_t", "grupo_ingreso_t",
                                     "n_activos_estables_t", "n_activos_estables_th", "share_precario_t",
                                     "trabajo_d16", "trabajo_lexicografico", "persiste_principal", "persiste_robustez"]
                salida = tabla_pares[columnas_targets].copy()
                salida["h"] = h
                salida["ANO4_t"], salida["TRIMESTRE_t"] = anio_t, trimestre_t
                salida["ANO4_th"], salida["TRIMESTRE_th"] = anio_th, trimestre_th
                salida.to_parquet(partition_path, index=False)

duracion_cascada = time.time() - t0
print(f"{len(filas)} filas de cascada_trimestral, {n_reusados} orígenes x h reusados de una corrida previa, "
      f"{duracion_cascada:.1f}s")

cascada_trimestral = pd.DataFrame(filas)
cascada_trimestral.to_csv(CASCADA_CSV, index=False)
print(f"escrito {CASCADA_CSV} -- {len(cascada_trimestral)} filas, {len(cascada_trimestral.columns)} columnas")
columnas_ingreso = [c for c in cascada_trimestral.columns if "ingreso" in c]
print(f"columnas del paso de ingreso ({len(columnas_ingreso)}): {columnas_ingreso}")
print("presentes en TODAS las filas (incluidas sin_destino_publicado, con NA donde no aplica):",
      all(c in cascada_trimestral.columns for c in columnas_ingreso))


## Tabla cruzada D24 — D16 tal cual vs. lectura lexicográfica

Sobre la población Y1 completa (86 trimestres × 2 horizontes, tratamiento
`categoria_propia`, **sin** excluir ingreso no declarado -- es la misma
población que ahora usa `cascada_trimestral.csv` por default), separado
techo (`share_precario_t==1.0`) de margen: fila = comparación cruda de
`share_precario` (`"sube"`/`"igual"`/`"baja"`, sin desempate); columna =
lectura final de 3 niveles. `"sube"` mapea siempre a `"empeora"` y `"baja"`
siempre a `"mejora"` -- el desempate solo actúa sobre la fila `"igual"` (la
prueba de que no inventa transiciones fuera de los empates reales).

In [ ]:
frames_pares = [pd.read_parquet(p) for p in TARGETS.rglob("data.parquet")]
pares_targets_completo = pd.concat(frames_pares, ignore_index=True)
confirmados = pares_targets_completo[pares_targets_completo["identidad_confirmada"] == True]
con_activos_total = confirmados[
    (confirmados["n_activos_estables_t"] > 0) & (confirmados["n_activos_estables_th"] > 0)].copy()
con_activos_total["techo"] = con_activos_total["share_precario_t"] == 1.0

for techo_val, nombre in ((True, "TECHO (share_precario_t == 1.0)"), (False, "MARGEN (share_precario_t < 1.0)")):
    sub = con_activos_total[con_activos_total["techo"] == techo_val]
    tabla = pd.crosstab(sub["trabajo_d16"], sub["trabajo_lexicografico"])
    tabla = tabla.reindex(index=["empeora", "igual", "mejora"], columns=["empeora", "igual", "mejora"], fill_value=0)
    tabla["TOTAL_fila"] = tabla.sum(axis=1)
    tabla.loc["TOTAL_columna"] = tabla.sum(axis=0)
    print(f"--- {nombre}, n={len(sub)} ---")
    display(tabla)

n_cambia_a_empeora = int(
    ((con_activos_total["trabajo_d16"] != "empeora") & (con_activos_total["trabajo_lexicografico"] == "empeora")).sum())
print(f"pares que pasan de \"no empeora\" (D16 tal cual) a \"empeora\" (lexicográfica): "
      f"{n_cambia_a_empeora:,} de {len(con_activos_total):,} ({100*n_cambia_a_empeora/len(con_activos_total):.2f}%)")

print()
print("--- grupo_ingreso_t en la población con activos estables (confirma la categoría no_declarado) ---")
display(con_activos_total["grupo_ingreso_t"].value_counts(dropna=False))


## Sección 3 (verificación) — reconciliación contra D9/D10/D16, y efecto de excluir ingreso

Los totales de `cascada_trimestral.csv` (tratamiento `categoria_propia`,
sin filtrar por era) tienen que reproducir `panel_resumen_pares.csv`/D10 en
`n_pares`/`n_identidad_confirmada`. La población Y1 (sin excluir ingreso,
el default desde esta ronda) reproduce D16 **después** de excluir los
orígenes D13 en `h=4` histórica (único ajuste que sigue haciendo falta --
D16 ya excluía esos 3 orígenes). Se muestra también cuánto baja la
población si se aplica la exclusión de ingreso no declarado
(`..._excl_ingreso`), para dimensionar el efecto sin aplicarlo por
default.

In [ ]:
cp = cascada_trimestral[cascada_trimestral["tratamiento_independiente"] == "categoria_propia"]
print("--- n_pares por h (D10/panel_resumen_pares.csv: 614.438 / 508.568) ---")
print(cp.groupby("h")["n_pares"].sum())
print()
print("--- n_identidad_confirmada por h (D10: 554.574 / 450.251) ---")
print(cp.groupby("h")["n_identidad_confirmada"].sum())
print()
print("--- población Y1 (sin excluir ingreso) por h x era, SIN excluir D13 ---")
print(cp.groupby(["h", "era"])["n_poblacion_y1"].sum())
print()
poblacion_y1_sin_d13 = cp[cp["corte_estructural_d13"] == False].groupby(["h", "era"])["n_poblacion_y1"].sum()
print("--- población Y1 (sin excluir ingreso) por h x era, excluyendo D13 "
      "(reconcilia exacto con D16: 241.033 / 188.131 / 184.587 / 152.419) ---")
print(poblacion_y1_sin_d13)
print()
poblacion_y1_excl_ingreso = cp[cp["corte_estructural_d13"] == False].groupby(["h", "era"])["n_poblacion_y1_excl_ingreso"].sum()
print("--- misma población, EXCLUYENDO además ingreso no declarado (reservada para la grilla 3x3) ---")
print(poblacion_y1_excl_ingreso)


## Registro de la corrida (Secciones 1-4)

Mismo patrón que el resto del pipeline (`pipeline_meta.registrar_corrida`,
`data/meta/corridas.csv`) -- fila aparte para esta sección, separada de la
de frecuencias/folds (Secciones 5-6, que corren su propio loop).

In [ ]:
pm.registrar_corrida(
    META / "corridas.csv", "07_cascada_targets_cascada",
    {"TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None, "REHACER": REHACER},
    len(trimestres_a_procesar), len(cascada_trimestral), duracion_cascada, COMMIT_GIT,
)
print(f"corrida de cascada_trimestral registrada -- {len(cascada_trimestral):,} filas, {duracion_cascada:.1f}s")


## Sección 5 — `vivienda_frecuencias.csv`

`cascada.frecuencias_vivienda` sobre `data/01_armonizado/hogar` (valores
crudos, antes de la capa tipada) -- transversal por trimestre, no necesita
pares. Incluye el chequeo pedido de si el bloque `IV*`/`II*` cambia de
códigos en el quiebre histórico/regular (D7, 2015T2→2016T2) -- D2 ya
verificó que no cambia en 2023T4, esto cubre el otro corte de era. Una
diferencia en el conjunto de valores *observados* entre dos trimestres
puntuales no prueba por sí sola un cambio de codificación (puede ser solo
que un código raro no apareció ese trimestre) -- se lista para revisión, no
se interpreta como redefinición confirmada. Después del chequeo crudo, se
contrasta cada variable marcada contra el diseño de registro (p.7-9) y
contra si `perfil_hogar.calcular_componentes_ancla` la usa.

In [ ]:
t0 = time.time()
filas_frecuencias, filas_frecuencias_agl = [], []
for anio, trimestre in trimestres_a_procesar:
    p = ARMONIZADO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    if not p.exists():
        continue
    hogar = pd.read_parquet(p, columns=["CODUSU", "NRO_HOGAR", "ANO4", "TRIMESTRE", "AGLOMERADO", *COLUMNAS_VIVIENDA])
    filas_frecuencias.append(frecuencias_vivienda(hogar, COLUMNAS_VIVIENDA))
    filas_frecuencias_agl.append(frecuencias_vivienda(hogar, COLUMNAS_VIVIENDA, por_aglomerado=True))

vivienda_frecuencias = pd.concat(filas_frecuencias, ignore_index=True)
vivienda_frecuencias_aglomerado = pd.concat(filas_frecuencias_agl, ignore_index=True)
vivienda_frecuencias.to_csv(META / f"vivienda_frecuencias{SUFIJO}.csv", index=False)
vivienda_frecuencias_aglomerado.to_csv(META / f"vivienda_frecuencias_aglomerado{SUFIJO}.csv", index=False)
duracion_frecuencias = time.time() - t0
print(f"vivienda_frecuencias: {len(vivienda_frecuencias)} filas, "
      f"vivienda_frecuencias_aglomerado: {len(vivienda_frecuencias_aglomerado)} filas, {duracion_frecuencias:.1f}s")

codigos_2015t2 = vivienda_frecuencias[(vivienda_frecuencias["ANO4"] == 2015) & (vivienda_frecuencias["TRIMESTRE"] == 2)]
codigos_2016t2 = vivienda_frecuencias[(vivienda_frecuencias["ANO4"] == 2016) & (vivienda_frecuencias["TRIMESTRE"] == 2)]
diferencias_d7 = []
for variable in COLUMNAS_VIVIENDA:
    valores_2015 = set(codigos_2015t2.loc[codigos_2015t2["variable"] == variable, "valor"].dropna())
    valores_2016 = set(codigos_2016t2.loc[codigos_2016t2["variable"] == variable, "valor"].dropna())
    if valores_2015 != valores_2016:
        diferencias_d7.append((variable, sorted(valores_2015), sorted(valores_2016)))
print(f"{len(diferencias_d7)} variables con distinto conjunto de valores observados, 2015T2 vs 2016T2:")
for variable, v2015, v2016 in diferencias_d7:
    usada_por_ancla = variable in ("II1", "II2", "II5", "II5_1", "IV1", "IV3", "IV8", "IV10", "IV12_1", "IV12_2", "IV12_3")
    print(f"  {variable}: {v2015} vs {v2016} -- usada por calcular_componentes_ancla: {usada_por_ancla}")


**Verificación contra el diseño de registro (p.7-9 del codebook), de las 6
variables marcadas arriba -- reporte, no se corrige nada de `perfil_hogar.py`**:

- **`IV2`** (N(2), "¿Cuántos ambientes/habitaciones tiene la vivienda en
  total?"): el código nuevo es `99`. Para un campo numérico de 2 dígitos,
  `99` es el código general de Ns/Nr (`CODIGOS_ESPECIALES_GENERALES` de
  `tipado_nucleo.py`: "9, 99, 999, 9999 según longitud del campo") -- no es
  una categoría nueva, es la no-respuesta estándar. **No la usa
  `calcular_componentes_ancla`** (la función no lee `IV2`) -- sin impacto
  en el ancla.
- **`IV9`** (N(1), "¿El baño o letrina está...? 1=dentro / 2=fuera, dentro
  del terreno / 3=fuera del terreno"): el código `3` **ya está en el
  diseño de registro** -- es una de las 3 categorías válidas, no una
  agregada en 2016. Que no apareciera en 2015T2 es ausencia muestral de una
  categoría rara en un trimestre puntual, no una redefinición. **No la usa
  `calcular_componentes_ancla`** (la función lee `IV8`/`IV10`, no `IV9`) --
  sin impacto en el ancla.
- **`II3_1`** (N(1), "si usa alguno exclusivamente como lugar de trabajo,
  ¿cuántos?"): es un conteo: el máximo observado sube de 4 (2015T2) a 6
  (2016T2) -- esperable por variación muestral en un conteo sin tope fijo
  en el diseño de registro, no hay código especial documentado más allá del
  `0`="no corresponde" general. **No la usa `calcular_componentes_ancla`**.
- **`II5_1`** (N(2), "si usa alguno para dormir, ¿cuántos?"): mismo caso,
  conteo sin tope fijo -- el código `3` es una respuesta real (3
  ambientes), no un código especial. **Sí la usa** `calcular_componentes_ancla`,
  pero solo para `hacinamiento_para_dormir` (`cuartos_dormir = II2 + II5_1`
  cuando `II5==1`) -- una columna que la función calcula y devuelve, pero
  que **no entra en `ancla_con_deficit`** (el flag combinado usa
  `hacinamiento_nbi`, `calidad_materiales_deficitaria` y
  `condiciones_sanitarias_deficitarias`, no `hacinamiento_para_dormir`). Un
  valor de `II5_1` más alto en 2016 no cambia `ancla_con_deficit` de ningún
  hogar, solo (potencialmente) `hacinamiento_para_dormir`, que se reporta
  aparte.
- **`II1`, `II2`** (conteos de ambientes, N(2) cada uno): mismo patrón que
  `II3_1`/`II5_1` -- el rango observado varía por trimestre sin que el
  diseño de registro cambie. `II1` sí entra en `ancla_con_deficit`
  (`hacinamiento_nbi = n_miembros/II1 > 3`) pero como conteo plano, sin
  código especial que dependa de si el valor máximo observado es 10 o 14 en
  un trimestre puntual.

**Conclusión**: de las 6 variables marcadas, ninguna tiene una redefinición
de código real en el diseño de registro entre 2015T2 y 2016T2 -- son 2
códigos de Ns/Nr/categoría rara que no habían aparecido en el trimestre
puntual de comparación (`IV2=99`, `IV9=3`) y 4 conteos sin tope fijo cuya
variación de rango es muestral. De las dos que sí alimentan
`calcular_componentes_ancla` (`II1`, `II5_1`), ninguna afecta
`ancla_con_deficit` -- `II5_1` solo llega a `hacinamiento_para_dormir`
(reportado aparte) y `II1` se usa como conteo plano sin códigos especiales
en juego. **No se modifica `perfil_hogar.py`.**

## Sección 6 — `cascada_folds.csv` (walk-forward con embargo)

`cascada.clasificar_fold` sobre el Parquet intermedio (tratamiento default,
`categoria_propia` -- alcanza para dimensionar casos, no hace falta repetir
por las 4 variantes acá), con `embargo=1` trimestre, **sin excluir** ingreso
no declarado (mismo default que la Sección 3 -- la categoría
`grupo_ingreso_t="no_declarado"` queda visible como una celda más del
desglose, en vez de restada antes de contar).

Folds de desarrollo: un test por año 2008-2022, eligiendo el **primer
trimestre del año con destino válido para `h=1` y `h=4` a la vez** (para
poder comparar ambos horizontes en el mismo fold) -- normalmente T1, salvo
2016 (T1 no está publicado, el primero válido para los dos horizontes es
T2) y 2015 (**ningún** trimestre de 2015 tiene destino válido para `h=1` y
`h=4` simultáneamente: T1 vale para `h=1` pero no para `h=4` -- destino
2016T1, en el hueco --, T2 no vale ni para `h=1` -- destino 2015T3, en el
hueco --, T3/T4 no están publicados o su destino tampoco lo está -- se
descarta 2015 en vez de forzar un trimestre que solo sirve para un
horizonte).

Holdout: test = todo origen desde 2023T1 (D19, uso único) hasta el último
outcome publicado; train = pares cuyo `t+h` cae hasta 2022T4 (mismo embargo
de 1 trimestre). Cada fila es `fold × h × partición × (ancla_con_deficit_t,
grupo_ingreso_t)`, con `celda_chica=True` si Y1 o Y2 tienen menos de 100
positivos en esa celda. `positivos_y2` = `sale_por_empleo` (variante
*principal*, D24 -- la lectura primaria de Y2, no `persiste`).

In [ ]:
t0 = time.time()
frames_resueltos = [pd.read_parquet(p) for p in TARGETS.rglob("data.parquet")]
pares_resueltos = pd.concat(frames_resueltos, ignore_index=True)
pares_resueltos = pares_resueltos[pares_resueltos["identidad_confirmada"] == True].copy()
print(f"pares elegibles para folds (identidad confirmada, SIN excluir ingreso no declarado): {len(pares_resueltos):,}")


def primer_trimestre_valido_para_ambos_h(anio: int) -> int | None:
    for trimestre in (1, 2, 3, 4):
        if (anio, trimestre) not in set(TODOS_LOS_TRIMESTRES):
            continue
        if tiene_destino_valido(anio, trimestre, 1) and tiene_destino_valido(anio, trimestre, 4):
            return trimestre
    return None


FOLDS_DEV = []
for anio in range(2008, 2023):
    trimestre = primer_trimestre_valido_para_ambos_h(anio)
    if trimestre is None:
        print(f"  {anio}: sin trimestre válido para h=1 y h=4 a la vez -- se descarta del fold de desarrollo")
        continue
    FOLDS_DEV.append((anio, trimestre))
print(f"folds de desarrollo: {FOLDS_DEV}")

HOLDOUT_DESDE = (2023, 1)


def resumen_poblacion(parte: pd.DataFrame) -> dict:
    pob_y1 = parte[(parte["n_activos_estables_t"] > 0) & (parte["n_activos_estables_th"] > 0)]
    pob_y2 = parte[parte["persiste_principal"].notna()]
    # positivos_y2 = sale_por_empleo (variante principal, D24) -- complemento exacto
    # de persiste_principal, mismo denominador (persiste_principal == False).
    return {
        "n_y1": len(pob_y1), "positivos_y1": int((pob_y1["trabajo_lexicografico"] == "empeora").sum()),
        "n_y2": len(pob_y2), "positivos_y2": int((pob_y2["persiste_principal"] == False).sum()),
    }


def filas_resumen(h, tipo_fold, anio_test, trimestre_test, particion, parte):
    filas_celda = [{"h": h, "tipo_fold": tipo_fold, "anio_test": anio_test, "trimestre_test": trimestre_test,
                     "particion": particion, "ancla_con_deficit_t": "TODOS", "grupo_ingreso_t": "TODOS",
                     **resumen_poblacion(parte)}]
    for (ancla_val, grupo_val), celda in parte.groupby(["ancla_con_deficit_t", "grupo_ingreso_t"], dropna=False):
        filas_celda.append({"h": h, "tipo_fold": tipo_fold, "anio_test": anio_test, "trimestre_test": trimestre_test,
                             "particion": particion, "ancla_con_deficit_t": ancla_val, "grupo_ingreso_t": grupo_val,
                             **resumen_poblacion(celda)})
    return filas_celda


filas_folds = []
for h in (1, 4):
    sub = pares_resueltos[pares_resueltos["h"] == h]

    for anio_test, trimestre_test in FOLDS_DEV:
        etiqueta = clasificar_fold(sub, anio_test, trimestre_test, embargo=1)
        for particion in ("train", "test"):
            filas_folds.extend(filas_resumen(h, "dev", anio_test, trimestre_test, particion, sub[etiqueta == particion]))

    es_test_holdout = (sub["ANO4_t"] > HOLDOUT_DESDE[0]) | (
        (sub["ANO4_t"] == HOLDOUT_DESDE[0]) & (sub["TRIMESTRE_t"] >= HOLDOUT_DESDE[1]))
    limite_anio, limite_trimestre = panel.sumar_trimestres(*HOLDOUT_DESDE, -1)
    es_train_holdout = (sub["ANO4_th"] < limite_anio) | (
        (sub["ANO4_th"] == limite_anio) & (sub["TRIMESTRE_th"] <= limite_trimestre))
    filas_folds.extend(filas_resumen(h, "holdout", *HOLDOUT_DESDE, "train", sub[es_train_holdout]))
    filas_folds.extend(filas_resumen(h, "holdout", *HOLDOUT_DESDE, "test", sub[es_test_holdout]))

cascada_folds = pd.DataFrame(filas_folds)
cascada_folds["celda_chica"] = (cascada_folds["positivos_y1"] < 100) | (cascada_folds["positivos_y2"] < 100)
cascada_folds.to_csv(META / f"cascada_folds{SUFIJO}.csv", index=False)
duracion_folds = time.time() - t0
n_celdas_chicas = int(cascada_folds["celda_chica"].sum())
print(f"cascada_folds: {len(cascada_folds)} filas, {n_celdas_chicas} celdas chicas, {duracion_folds:.1f}s")

print("--- folds de desarrollo (TODOS, test) -- ya no debería haber ningún N=0 estructural ---")
dev_test = cascada_folds[(cascada_folds["tipo_fold"] == "dev") & (cascada_folds["particion"] == "test")
                          & (cascada_folds["ancla_con_deficit_t"] == "TODOS")]
display(dev_test[["h", "anio_test", "trimestre_test", "n_y1", "positivos_y1", "n_y2", "positivos_y2"]])

print("--- línea de desarrollo (min/mediana/máx de folds dev) vs. holdout, agregado (TODOS) ---")
agregado = cascada_folds[(cascada_folds["ancla_con_deficit_t"] == "TODOS") & (cascada_folds["particion"] == "test")]
display(agregado.groupby(["h", "tipo_fold"])[["n_y1", "positivos_y1", "n_y2", "positivos_y2"]].agg(
    ["min", "median", "max"]))


## Registro de la corrida (Secciones 5-6)

In [ ]:
duracion_total_5_6 = duracion_frecuencias + duracion_folds
pm.registrar_corrida(
    META / "corridas.csv", "07_cascada_targets_frecuencias_folds",
    {"TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None},
    len(trimestres_a_procesar), len(vivienda_frecuencias) + len(cascada_folds), duracion_total_5_6, COMMIT_GIT,
)
print(f"corrida de frecuencias+folds registrada -- {duracion_total_5_6:.1f}s")
print(f"notebook completa -- {time.time()-t0_proceso:.1f}s en total")
